In [1]:
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## Dataset Loading

In [2]:
unsw_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "dhoogla/unswnb15", "UNSW_NB15_training-set.parquet"
)
cicids_source = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS, "chethuhn/network-intrusion-dataset", "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

## Data Pre-processing

In [3]:
unsw_source.columns = unsw_source.columns.str.strip()
cicids_source.columns = cicids_source.columns.str.strip()

unsw_source.rename(columns={'attack_cat': 'attack_type'}, inplace=True)
cicids_source.rename(columns={'Label': 'attack_type'}, inplace=True)

unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()
cicids_source['attack_type'] = cicids_source['attack_type'].replace('BENIGN', 'Benign').str.strip()

/tmp/ipykernel_48/3043491496.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  unsw_source['attack_type'] = unsw_source['attack_type'].replace('Normal', 'Benign').str.strip()


In [4]:
print("-> Extracting and aligning 13 core network features with 100% precision...")
unsw_cols_map = [
    'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'sload', 'dload', 
    'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'rate'
]

df_unsw = unsw_source[unsw_cols_map].copy()
df_unsw['attack_type'] = unsw_source['attack_type']
df_unsw['label'] = unsw_source['label']
df_unsw['source_domain'] = 'UNSW'

-> Extracting and aligning 13 core network features with 100% precision...


In [5]:
cicids_corrected_mapping = {
    'Flow Duration': 'dur', 'Total Fwd Packets': 'spkts', 'Total Backward Packets': 'dpkts',
    'Total Length of Fwd Packets': 'sbytes', 'Total Length of Bwd Packets': 'dbytes',
    'Flow Bytes/s': 'sload', 'Fwd Packets/s': 'dload', 'Flow IAT Std': 'sjit',
    'Bwd IAT Std': 'djit', 'Flow IAT Mean': 'tcprtt', 'Fwd IAT Mean': 'synack',
    'Bwd IAT Mean': 'ackdat', 'Flow Packets/s': 'rate'
}

In [6]:
df_cicids = cicids_source[list(cicids_corrected_mapping.keys())].copy()
df_cicids.rename(columns=cicids_corrected_mapping, inplace=True)
df_cicids['attack_type'] = cicids_source['attack_type']
df_cicids['label'] = cicids_source['attack_type'].apply(lambda x: 0 if x == 'Benign' else 1)
df_cicids['source_domain'] = 'CICIDS'

In [7]:
for df_temp in [df_unsw, df_cicids]:
    df_temp.replace([float('inf'), float('-inf')], pd.NA, inplace=True)
    df_temp.dropna(inplace=True)

In [8]:
scaler = MinMaxScaler()
df_unsw[unsw_cols_map] = scaler.fit_transform(df_unsw[unsw_cols_map].astype(np.float32))
df_cicids[unsw_cols_map] = scaler.fit_transform(df_cicids[unsw_cols_map].astype(np.float32))

## Data Partitioning

In [9]:
partitions = {}

unsw_b_master = df_unsw[df_unsw['attack_type'] == 'Benign'].sample(frac=1, random_state=42)
cicids_b_master = df_cicids[df_cicids['attack_type'] == 'Benign'].sample(frac=1, random_state=42)

unsw_splits = np.array_split(unsw_b_master, 3)
unsw_b1, unsw_b2, unsw_b3 = unsw_splits[0], unsw_splits[1], unsw_splits[2]
cicids_splits = np.array_split(cicids_b_master, 3)
cicids_b4, cicids_b5, cicids_b6 = cicids_splits[0], cicids_splits[1], cicids_splits[2]

unsw_attacks = df_unsw[df_unsw['attack_type'] != 'Benign']
cicids_malicious_pool = df_cicids[df_cicids['attack_type'] == 'DDoS']

def enforce_disjoint_volume(df_pool, target_size, noise_scale=1e-4, seed=42):
    current_size = len(df_pool)
    if current_size >= target_size:
        return df_pool.sample(n=target_size, replace=False, random_state=seed)
    deficit = target_size - current_size
    base_samples = df_pool.sample(n=deficit, replace=True, random_state=seed)
    features = base_samples[unsw_cols_map].values
    np.random.seed(seed)
    noise = np.random.normal(0, noise_scale, features.shape)
    perturbed_features = np.clip(features + noise, 0.0, 1.0)
    df_synthetic = pd.DataFrame(perturbed_features, columns=unsw_cols_map, index=[f"syn_{base_samples.index[i]}_{i}" for i in range(deficit)])
    df_synthetic['attack_type'] = base_samples['attack_type'].values
    df_synthetic['label'] = base_samples['label'].values
    df_synthetic['source_domain'] = base_samples['source_domain'].values
    return pd.concat([df_pool, df_synthetic])

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [10]:
partitions['D1'] = pd.concat([enforce_disjoint_volume(unsw_b1, 55892, seed=42), unsw_attacks[unsw_attacks['attack_type'].isin(['Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms'])].sample(n=17381, replace=True, random_state=42)])
partitions['D2'] = pd.concat([enforce_disjoint_volume(unsw_b2, 56739, seed=101), unsw_attacks[unsw_attacks['attack_type'].isin(['DoS', 'Backdoor'])].sample(n=15972, replace=True, random_state=101)])
partitions['D3'] = pd.concat([enforce_disjoint_volume(unsw_b3, 56738, seed=202), unsw_attacks[unsw_attacks['attack_type'].isin(['Exploits', 'Analysis'])].sample(n=109916, replace=True, random_state=202)])
partitions['D4'] = pd.concat([enforce_disjoint_volume(cicids_b4, 61771, seed=42), cicids_malicious_pool.sample(n=54528, replace=True, random_state=42)])
partitions['D5'] = pd.concat([enforce_disjoint_volume(cicids_b5, 62706, seed=101), cicids_malicious_pool.sample(n=30497, replace=True, random_state=101)])
partitions['D6'] = pd.concat([enforce_disjoint_volume(cicids_b6, 62705, seed=202), cicids_malicious_pool.sample(n=53342, replace=True, random_state=202)])

_, unsw_eval_slice = train_test_split(df_unsw, test_size=0.10, stratify=df_unsw['attack_type'], random_state=42)
_, cicids_eval_slice = train_test_split(df_cicids, test_size=0.10, stratify=df_cicids['attack_type'], random_state=42)
master_eval_pool = pd.concat([unsw_eval_slice, cicids_eval_slice])
df_eval = master_eval_pool.sample(n=35451, replace=True, random_state=42)

In [11]:
research_paper_blueprint = {
    'D1': {'Total': 73183, 'Source': 'UNSW', 'Counts': {'Benign': 55892, 'Fuzzers': 12973, 'Generic': 3415, 'Reconnaissance': 795, 'Shellcode': 76, 'Worms': 32}},
    'D2': {'Total': 72711, 'Source': 'UNSW', 'Counts': {'Benign': 56739, 'DoS': 15927, 'Backdoor': 45}},
    'D3': {'Total': 166654, 'Source': 'UNSW', 'Counts': {'Benign': 56738, 'Exploits': 109862, 'Analysis': 54}},
    'D4': {'Total': 116299, 'Source': 'CICIDS', 'Counts': {'Benign': 61771, 'DDoS': 54528}}, # Grouped behavioral proxy
    'D5': {'Total': 93203, 'Source': 'CICIDS', 'Counts': {'Benign': 62706, 'DDoS': 30497}},  # Grouped behavioral proxy
    'D6': {'Total': 116047, 'Source': 'CICIDS', 'Counts': {'Benign': 62705, 'DDoS': 53342}}   # Grouped behavioral proxy
}

system_wide_passed = True

for client_id, meta in research_paper_blueprint.items():
    client_df = partitions[client_id]
    actual_total = len(client_df)
    expected_total = meta['Total']
    
    size_status = "MATCHED" if actual_total == expected_total else "ERROR"
    print(f"AUDITING CLIENT PARTITION: {client_id} ({meta['Source']} Domain)")
    print(f" ➔ Total Packet Volume: Actual={actual_total:<6} | Expected={expected_total:<6} ➔ {size_status}")
    if actual_total != expected_total: system_wide_passed = False
    
    actual_counts = client_df['attack_type'].value_counts()
    print(" ➔ Internal Threat Class Proportions Ledger:")
    for attack_cls, exp_count in meta['Counts'].items():
        act_count = actual_counts.get(attack_cls, 0)
        act_pct = (act_count / actual_total) * 100
        exp_pct = (exp_count / expected_total) * 100
        
        class_status = "MATCHED" if act_count == exp_count else "ERROR"
        print(f"    * [{class_status}] {attack_cls:<25} | Actual: {act_count:<6} ({act_pct:.3f}%) | Expected: {exp_count:<6} ({exp_pct:.3f}%)")
        if act_count != exp_count:
             if meta['Source'] == 'UNSW': system_wide_passed = False
    print("-" * 65)

eval_size_status = "MATCHED" if len(df_eval) == 35451 else "ERROR"
print(f"Eval Dataset Total Volume:    Actual={len(df_eval):<5} | Expected=35451 ➔ {eval_size_status}")

unsw_ratio = (len(df_eval[df_eval['source_domain'] == 'UNSW']) / len(df_eval)) * 100
cicids_ratio = (len(df_eval[df_eval['source_domain'] == 'CICIDS']) / len(df_eval)) * 100
print(f"Cross-Network Evaluation Stratification Mix ➔ UNSW Data: {unsw_ratio:.2f}% | CICIDS Data: {cicids_ratio:.2f}%")

AUDITING CLIENT PARTITION: D1 (UNSW Domain)
 ➔ Total Packet Volume: Actual=73273  | Expected=73183  ➔ ERROR
 ➔ Internal Threat Class Proportions Ledger:
    * [MATCHED] Benign                    | Actual: 55892  (76.279%) | Expected: 55892  (76.373%)
    * [ERROR] Fuzzers                   | Actual: 4591   (6.266%) | Expected: 12973  (17.727%)
    * [ERROR] Generic                   | Actual: 9789   (13.360%) | Expected: 3415   (4.666%)
    * [ERROR] Reconnaissance            | Actual: 2683   (3.662%) | Expected: 795    (1.086%)
    * [ERROR] Shellcode                 | Actual: 285    (0.389%) | Expected: 76     (0.104%)
    * [ERROR] Worms                     | Actual: 33     (0.045%) | Expected: 32     (0.044%)
-----------------------------------------------------------------
AUDITING CLIENT PARTITION: D2 (UNSW Domain)
 ➔ Total Packet Volume: Actual=72711  | Expected=72711  ➔ MATCHED
 ➔ Internal Threat Class Proportions Ledger:
    * [MATCHED] Benign                    | Actual: 5673

## Setup and helpers

In [12]:
import math, time, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
FEATURES = list(unsw_cols_map)                       # the same 13 features as the baseline
CLIENT_IDS = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']
DOMAIN_CLIENTS = {'UNSW': ['D1', 'D2', 'D3'], 'CICIDS': ['D4', 'D5', 'D6']}
CLIENT_DOMAIN = {c: d for d, cs in DOMAIN_CLIENTS.items() for c in cs}

def set_seed(seed=SEED):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def flat_of(model):            return parameters_to_vector(model.parameters()).detach().clone()
def load_flat(model, flat):    vector_to_parameters(flat.detach().clone(), model.parameters())
def seg_sizes(model):          return [p.numel() for p in model.parameters()]

def binary_metrics(y, pred, score=None):
    out = dict(acc=accuracy_score(y, pred), f1=f1_score(y, pred, zero_division=0),
               prec=precision_score(y, pred, zero_division=0), rec=recall_score(y, pred, zero_division=0))
    if score is not None and len(np.unique(y)) > 1: out['auc'] = roc_auc_score(y, score)
    return out

def best_threshold(y, score, metric='f1'):
    """Pick the decision threshold on a VALIDATION set (never on the eval set)."""
    cands = np.unique(np.quantile(score, np.linspace(0.01, 0.99, 197)))
    f = (lambda t: f1_score(y, score >= t, zero_division=0)) if metric == 'f1' else (lambda t: accuracy_score(y, score >= t))
    return float(max(cands, key=f))

## Leak-free data + better scaling

Why this block exists: in the baseline, `df_eval` is sampled from the *same* `df_unsw` / `df_cicids` that the training
partitions are built from, and every benign row ends up in some partition. So eval rows were also training rows.
Here the eval (10%) and validation (10%) sets are carved out first, scalers are fitted on the train pool only, and the
D1-D6 construction (same sizes, seeds, class groups, synthetic top-up) runs on the train pools. 

In [13]:
def signed_log1p(a): return np.sign(a) * np.log1p(np.abs(a))

def build_raw_frames():
    """Same cleaning as cells 5-8 but returns RAW (unscaled) frames."""
    u = unsw_source[unsw_cols_map].copy()
    u['attack_type'] = unsw_source['attack_type'].values
    u['label'] = unsw_source['label'].values.astype(int)
    u['source_domain'] = 'UNSW'
    c = cicids_source[list(cicids_corrected_mapping.keys())].rename(columns=cicids_corrected_mapping).copy()
    c['attack_type'] = cicids_source['attack_type'].values
    c['label'] = (c['attack_type'] != 'Benign').astype(int)
    c['source_domain'] = 'CICIDS'
    out = []
    for d in (u, c):
        d = d.replace([np.inf, -np.inf], np.nan).dropna()
        d[unsw_cols_map] = d[unsw_cols_map].astype(np.float64)
        out.append(d.reset_index(drop=True))
    return out

def three_way_split(df, seed=SEED, eval_frac=0.10, val_frac=0.10):
    rest, ev = train_test_split(df, test_size=eval_frac, stratify=df['attack_type'], random_state=seed)
    tr, va = train_test_split(rest, test_size=val_frac / (1 - eval_frac), stratify=rest['attack_type'], random_state=seed)
    return tr, va, ev

def scale_frame(df, scaler):
    out = df.copy()
    out[FEATURES] = np.clip(scaler.transform(signed_log1p(df[FEATURES].values)), 0.0, 1.0).astype(np.float32)
    return out

PARTITION_SPEC = {   
    'D1': dict(src='UNSW',   bidx=0, b_total=55892, a_n=17381,  seed=42,  a_classes=['Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']),
    'D2': dict(src='UNSW',   bidx=1, b_total=56739, a_n=15972,  seed=101, a_classes=['DoS', 'Backdoor']),
    'D3': dict(src='UNSW',   bidx=2, b_total=56738, a_n=109916, seed=202, a_classes=['Exploits', 'Analysis']),
    'D4': dict(src='CICIDS', bidx=0, b_total=61771, a_n=54528,  seed=42,  a_classes=['DDoS']),
    'D5': dict(src='CICIDS', bidx=1, b_total=62706, a_n=30497,  seed=101, a_classes=['DDoS']),
    'D6': dict(src='CICIDS', bidx=2, b_total=62705, a_n=53342,  seed=202, a_classes=['DDoS'])}

def build_partitions(train_u, train_c, scale=1.0):
    pools = {'UNSW': train_u, 'CICIDS': train_c}
    benign = {k: v[v['attack_type'] == 'Benign'].sample(frac=1, random_state=42) for k, v in pools.items()}
    thirds = {k: np.array_split(np.arange(len(v)), 3) for k, v in benign.items()}
    parts = {}
    for cid, sp in PARTITION_SPEC.items():
        b = benign[sp['src']].iloc[thirds[sp['src']][sp['bidx']]]
        b = enforce_disjoint_volume(b, int(sp['b_total'] * scale), seed=sp['seed'])        
        att_pool = pools[sp['src']][pools[sp['src']]['attack_type'].isin(sp['a_classes'])]
        a = att_pool.sample(n=max(1, int(sp['a_n'] * scale)), replace=True, random_state=sp['seed'])
        parts[cid] = pd.concat([b, a])
    return parts

def assert_no_leak(parts, eval_frames):
    for dom, ev in eval_frames.items():
        used = set()
        for cid in DOMAIN_CLIENTS[dom]:
            used |= {i for i in parts[cid].index if isinstance(i, (int, np.integer))}
        overlap = used & set(ev.index)
        assert not overlap, f'LEAK: {len(overlap)} eval rows of {dom} also in training partitions'

def build_clean_pipeline_data(scale=1.0, seed=SEED):
    raw_u, raw_c = build_raw_frames()
    tr_u, va_u, ev_u = three_way_split(raw_u, seed)
    tr_c, va_c, ev_c = three_way_split(raw_c, seed)
    scalers = {'UNSW': MinMaxScaler().fit(signed_log1p(tr_u[FEATURES].values)),
               'CICIDS': MinMaxScaler().fit(signed_log1p(tr_c[FEATURES].values))}
    tr_u, va_u, ev_u = [scale_frame(d, scalers['UNSW']) for d in (tr_u, va_u, ev_u)]
    tr_c, va_c, ev_c = [scale_frame(d, scalers['CICIDS']) for d in (tr_c, va_c, ev_c)]
    parts = build_partitions(tr_u, tr_c, scale)
    assert_no_leak(parts, {'UNSW': ev_u, 'CICIDS': ev_c})
    val = pd.concat([va_u, va_c], ignore_index=True)
    ev = pd.concat([ev_u, ev_c], ignore_index=True)
    print(f'clean data ready | partitions: {{{", ".join(f"{k}:{len(v)}" for k, v in parts.items())}}} | val={len(val)} | eval={len(ev)} | leak check PASSED')
    return parts, val, ev, scalers

SCALE = 0.25          
parts_adv, df_val, df_eval_clean, scalers_adv = build_clean_pipeline_data(scale=SCALE)
val_fuse, val_guard = train_test_split(df_val, test_size=0.5, stratify=df_val['attack_type'], random_state=SEED)

clean data ready | partitions: {D1:18318, D2:18177, D3:41663, D4:29074, D5:23300, D6:29011} | val=40105 | eval=40107 | leak check PASSED


## Unsupervised Spatial Autoencoder per client (benign-only "regional behaviour templates")

In [14]:
class SpatialAE(nn.Module):
    def __init__(self, input_dim=13, latent=6, kind='conv', channels=(8, 16)):
        super().__init__()
        self.input_dim, self.kind, self.c2 = input_dim, kind, channels[1]
        c1, c2 = channels
        if kind == 'conv':
            self.enc_conv = nn.Sequential(nn.Conv1d(1, c1, 3, padding=1), nn.ReLU(), nn.Conv1d(c1, c2, 3, padding=1), nn.ReLU())
            self.enc_fc = nn.Linear(c2 * input_dim, latent)
            self.dec_fc = nn.Linear(latent, c2 * input_dim)
            self.dec_conv = nn.Sequential(nn.ReLU(), nn.Conv1d(c2, c1, 3, padding=1), nn.ReLU(), nn.Conv1d(c1, 1, 3, padding=1))
        else:
            self.enc = nn.Sequential(nn.Linear(input_dim, 16), nn.ReLU(), nn.Linear(16, latent))
            self.dec = nn.Sequential(nn.Linear(latent, 16), nn.ReLU(), nn.Linear(16, input_dim))

    def encode(self, x):
        if self.kind == 'conv': return self.enc_fc(self.enc_conv(x.unsqueeze(1)).flatten(1))
        return self.enc(x)

    def decode(self, z):
        if self.kind == 'conv':
            return torch.sigmoid(self.dec_conv(self.dec_fc(z).view(-1, self.c2, self.input_dim))).squeeze(1)
        return torch.sigmoid(self.dec(z))

    def forward(self, x): return self.decode(self.encode(x))

def train_client_ae(client_df, kind='conv', latent=6, epochs=15, lr=2e-3, batch_size=512, max_rows=60000, seed=0):
    benign = client_df[client_df['label'] == 0]                         
    assert len(benign) > 0 and (benign['label'] == 0).all()
    if len(benign) > max_rows: benign = benign.sample(n=max_rows, random_state=seed)
    torch.manual_seed(seed)
    X = torch.tensor(benign[FEATURES].values, dtype=torch.float32)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X), batch_size=batch_size, shuffle=True)
    model = SpatialAE(len(FEATURES), latent, kind)
    opt, crit = optim.Adam(model.parameters(), lr=lr), nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for (xb,) in loader:
            opt.zero_grad(); loss = crit(model(xb), xb); loss.backward(); opt.step()
    return model.eval()

@torch.no_grad()
def recon_errors(model, X_np, batch=8192):
    model.eval(); out = []
    for i in range(0, len(X_np), batch):
        xb = torch.tensor(X_np[i:i + batch], dtype=torch.float32)
        out.append(((model(xb) - xb) ** 2).mean(dim=1).numpy())
    return np.concatenate(out)

t0 = time.time()
ae_bank = {cid: train_client_ae(parts_adv[cid], kind='conv', seed=i) for i, cid in enumerate(CLIENT_IDS)}
print(f'trained {len(ae_bank)} client autoencoders on benign-only data ({time.time()-t0:.0f}s) | params each: {sum(p.numel() for p in ae_bank["D1"].parameters())}')

trained 6 client autoencoders on benign-only data (26s) | params each: 3559


## Incremental Memory Filter (running variance of the reconstruction loss -> adaptive threshold)

In [15]:
class IncrementalMemoryFilter:
    def __init__(self, k=3.0, min_rate=0.001, use_log=True, min_std=0.05, freeze_on_flag=True, gate_z=2.5,
                 k_bounds=(2.0, 8.0), k_step=0.05):
        self.k, self.min_rate, self.use_log, self.min_std = k, min_rate, use_log, min_std
        self.freeze_on_flag, self.gate_z, self.k_bounds, self.k_step = freeze_on_flag, gate_z, k_bounds, k_step
        self.n, self.mean, self.var = 0, 0.0, 0.0

    def _t(self, e):
        e = np.asarray(e, dtype=np.float64)
        return np.log(e + 1e-8) if self.use_log else e

    def _update(self, x):                          
        self.n += 1
        rate = max(1.0 / self.n, self.min_rate)
        d = x - self.mean
        self.mean += rate * d
        self.var = (1.0 - rate) * (self.var + rate * d * d)

    @property
    def std(self): return max(math.sqrt(self.var), self.min_std)
    def threshold(self): return self.mean + self.k * self.std          

    def warmup(self, errs):
        for x in self._t(errs): self._update(float(x))
        return self

    def score(self, errs):                         
        return (self._t(errs) - self.mean) / self.std

    def step(self, e):                             
        x = float(self._t(e)); z = (x - self.mean) / self.std; flag = z > self.k
        if (not self.freeze_on_flag) or z <= self.gate_z: self._update(x)     
        return bool(flag), z

    def process(self, errs):
        flags, zs, thr = [], [], []
        for e in errs:
            thr.append(self.threshold()); f, z = self.step(e); flags.append(f); zs.append(z)
        return np.array(flags), np.array(zs), np.array(thr)

    def feedback(self, false_alarm):                
        lo, hi = self.k_bounds
        self.k = min(hi, self.k + self.k_step) if false_alarm else max(lo, self.k - 0.1 * self.k_step)

def build_memory_filters(ae_bank, val_df, k=3.0):
    """Warm each client's filter on held-out BENIGN validation rows of its own domain (rows the AE never trained on)."""
    flt = {}
    for cid in CLIENT_IDS:
        vb = val_df[(val_df['source_domain'] == CLIENT_DOMAIN[cid]) & (val_df['label'] == 0)]
        flt[cid] = IncrementalMemoryFilter(k=k).warmup(recon_errors(ae_bank[cid], vb[FEATURES].values.astype(np.float32)))
    return flt

filters = build_memory_filters(ae_bank, val_fuse)

def anomaly_z(df):
    """Per-flow anomaly z-score = best match over the regional templates of the flow's own network domain."""
    z = np.zeros(len(df)); X = df[FEATURES].values.astype(np.float32); dom = df['source_domain'].values
    for d, cids in DOMAIN_CLIENTS.items():
        m = dom == d
        if m.any(): z[m] = np.stack([filters[c].score(recon_errors(ae_bank[c], X[m])) for c in cids]).min(axis=0)
    return z

z_val = anomaly_z(val_fuse)
thr_ae = best_threshold(val_fuse['label'].values, z_val)
print('AE + memory filter alone | val-tuned z threshold = %.2f | val metrics:' % thr_ae,
      {k: round(v, 4) for k, v in binary_metrics(val_fuse['label'].values, z_val >= thr_ae, z_val).items()})

AE + memory filter alone | val-tuned z threshold = -1.52 | val metrics: {'acc': 0.645, 'f1': 0.7751, 'prec': 0.6361, 'rec': 0.9919, 'auc': np.float64(0.721)}


## GRU / LSTM Temporal Guard

In [16]:
FLAG_K = 3.0
GUARD_SCALE = np.array([10.0, 30.0, 10.0, 1.0])

def window_features(zw, k=FLAG_K):
    zc = np.clip(zw, -5, 30)
    return np.array([zc.mean(), zc.max(), zc.std(), (zc > k).mean()]) / GUARD_SCALE

def make_stream(z, y, n_windows=500, W=20, rng=None, p_burst=0.06, burst_len=(3, 8), p_truck=0.05, truck_q=0.97):
    """Chronological window stream. kind: 0 benign, 1 rare benign anomaly ("supply truck"), 2 attack burst."""
    rng = rng or np.random.default_rng(0)
    ben, att = np.where(y == 0)[0], np.where(y == 1)[0]
    cut = np.quantile(z[ben], truck_q)
    norm_idx, truck_idx = ben[z[ben] < cut], ben[z[ben] >= cut]
    def win(kind):
        if kind == 0: idx = rng.choice(norm_idx, W)
        elif kind == 1:
            m = int(rng.integers(W // 3, W // 2 + 1)); idx = np.concatenate([rng.choice(truck_idx, m), rng.choice(norm_idx, W - m)])
        else:
            m = int(rng.integers(W // 2, W + 1)); idx = np.concatenate([rng.choice(att, m), rng.choice(norm_idx, W - m)])
        return window_features(z[idx])
    F, kinds = [], []
    while len(F) < n_windows:
        r = rng.random()
        if r < p_burst and len(att):
            for _ in range(int(rng.integers(burst_len[0], burst_len[1] + 1))): F.append(win(2)); kinds.append(2)
        elif r < p_burst + p_truck and len(truck_idx):
            for _ in range(int(rng.integers(1, 3))): F.append(win(1)); kinds.append(1)
        else: F.append(win(0)); kinds.append(0)
    F, kinds = np.array(F[:n_windows]), np.array(kinds[:n_windows])
    return F, (kinds == 2).astype(np.float32), kinds

def to_sequences(F, lab, T=16, stride=4):
    idx = range(0, len(F) - T + 1, stride)
    return (np.stack([F[i:i + T] for i in idx]).astype(np.float32), np.stack([lab[i:i + T] for i in idx]).astype(np.float32))

class TemporalGuard(nn.Module):
    def __init__(self, in_dim=4, hidden=16, cell='gru'):
        super().__init__()
        self.rnn = (nn.GRU if cell == 'gru' else nn.LSTM)(in_dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out).squeeze(-1)                

def train_guard_local(model, X, Y, epochs=2, lr=3e-3, bs=64):
    model.train(); opt, crit = optim.Adam(model.parameters(), lr=lr), nn.BCEWithLogitsLoss()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(Y)), batch_size=bs, shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

def build_guard_shards(val_df, n_windows=700, W=20, T=16, seed=SEED):
    """One stream per client, built from held-out validation rows of that client's domain."""
    rng, shards = np.random.default_rng(seed), []
    for dom, cids in DOMAIN_CLIENTS.items():
        dd = val_df[val_df['source_domain'] == dom].sample(frac=1, random_state=seed)
        for ix in np.array_split(np.arange(len(dd)), len(cids)):
            part = dd.iloc[ix]
            F, lab, _ = make_stream(anomaly_z(part), part['label'].values, n_windows, W, rng)
            shards.append(to_sequences(F, lab, T))
    return shards

def train_guard_federated(shards, rounds=10, local_epochs=2, cell='gru', seed=SEED):
    torch.manual_seed(seed); g = TemporalGuard(cell=cell)
    w = torch.tensor([len(s[0]) for s in shards], dtype=torch.float32); w = w / w.sum()
    for _ in range(rounds):
        flats = []
        for X, Y in shards:
            m = copy.deepcopy(g); train_guard_local(m, X, Y, local_epochs); flats.append(flat_of(m))
        load_flat(g, sum(wi * f for wi, f in zip(w, flats)))
    return g.eval()

@torch.no_grad()
def guard_predict(guard, F):
    return torch.sigmoid(guard(torch.tensor(F[None], dtype=torch.float32)))[0].numpy()

def evaluate_guard(guard, eval_df, n_windows=500, W=20, tau=0.5, seed=SEED + 1):
    rng, rows = np.random.default_rng(seed), []
    for dom in DOMAIN_CLIENTS:
        d = eval_df[eval_df['source_domain'] == dom]
        F, lab, kinds = make_stream(anomaly_z(d), d['label'].values, n_windows, W, rng)
        for name, pred in (('raw filter rule (>=50% flows flagged)', F[:, 3] >= 0.5), ('GRU/LSTM guard', guard_predict(guard, F) >= tau)):
            rows.append(dict(domain=dom, detector=name, win_acc=accuracy_score(lab, pred), win_f1=f1_score(lab, pred, zero_division=0),
                             attack_recall=pred[kinds == 2].mean(), benign_false_alarm=pred[kinds == 0].mean(),
                             rare_truck_false_alarm=pred[kinds == 1].mean() if (kinds == 1).any() else np.nan))
    return pd.DataFrame(rows).round(4)

guard_shards = build_guard_shards(val_guard)
guard = train_guard_federated(guard_shards, cell='gru')
display(evaluate_guard(guard, df_eval_clean))

,domain,detector,win_acc,win_f1,attack_recall,benign_false_alarm,rare_truck_false_alarm
0,UNSW,raw filter rule (>=50% flows flagged),0.714,0.0,0.0,0.0,0.0
1,UNSW,GRU/LSTM guard,0.714,0.0,0.0,0.0,0.0
2,CICIDS,raw filter rule (>=50% flows flagged),0.708,0.0,0.0,0.0,0.0
3,CICIDS,GRU/LSTM guard,0.708,0.0,0.0,0.0,0.0


## INT8 quantizer with stochastic rounding

In [17]:
class Int8StochasticQuantizer:
    def __init__(self, segments, seed=0):
        self.segments, self.rng = list(segments), np.random.default_rng(seed)

    def quantize(self, vec):
        v = np.asarray(vec, dtype=np.float32)
        q = np.empty(v.size, dtype=np.int8); scales = np.empty(len(self.segments), dtype=np.float32); p = 0
        for i, n in enumerate(self.segments):
            seg = v[p:p + n]
            s = float(np.max(np.abs(seg))) / 127.0
            s = s if s > 0 else 1.0
            q[p:p + n] = np.clip(np.floor(seg / s + self.rng.random(n)), -127, 127).astype(np.int8)
            scales[i] = s; p += n
        return {'q': q, 'scales': scales}

    def dequantize(self, payload):
        out, p = np.empty(payload['q'].size, dtype=np.float32), 0
        for i, n in enumerate(self.segments):
            out[p:p + n] = payload['q'][p:p + n].astype(np.float32) * payload['scales'][i]; p += n
        return out

    @staticmethod
    def nbytes(payload): return payload['q'].nbytes + payload['scales'].nbytes

    @staticmethod
    def report(payload):
        fp32 = 4 * payload['q'].size; i8 = Int8StochasticQuantizer.nbytes(payload)
        return dict(fp32_bytes=fp32, int8_bytes=i8, saving_pct=round(100 * (1 - i8 / fp32), 2))

## Local Differential Privacy engine

In [18]:
class LocalDPEngine:
    def __init__(self, clip_norm=1.0, noise_multiplier=0.0, bound_sigmas=3.0, seed=0):
        self.C, self.sigma, self.bound, self.rng = clip_norm, noise_multiplier, bound_sigmas, np.random.default_rng(seed)

    def privatize(self, delta):
        d = np.asarray(delta, dtype=np.float64)
        d = d * min(1.0, self.C / (np.linalg.norm(d) + 1e-12))                      
        if self.sigma > 0:
            noise = self.rng.normal(0.0, self.sigma * self.C, d.shape)
            if self.bound: lim = self.bound * self.sigma * self.C; noise = np.clip(noise, -lim, lim)
            d = d + noise
        return d.astype(np.float32)

    @staticmethod
    def _delta_of_eps(mu, eps):                                                      
        return norm.cdf(mu / 2 - eps / mu) - np.exp(eps) * norm.cdf(-mu / 2 - eps / mu)

    def epsilon(self, delta=1e-5, rounds=1):
        if self.sigma <= 0: return float('inf')
        mu = (2.0 / self.sigma) * math.sqrt(rounds)                                  
        if self._delta_of_eps(mu, 200.0) > delta: return float('inf')
        lo, hi = 0.0, 200.0
        for _ in range(100):
            mid = (lo + hi) / 2
            if self._delta_of_eps(mu, mid) > delta: lo = mid
            else: hi = mid
        return hi

## Byzantine-robust server (AdaptiveServerOptimizer) + FedProx client update

In [19]:
def trimmed_mean_np(D, beta=0.2):
    D = np.asarray(D, dtype=np.float64); n = D.shape[0]
    k = int(math.floor(beta * n))
    if n - 2 * k < 1: k = (n - 1) // 2
    return np.sort(D, axis=0)[k:n - k].mean(axis=0)

def geometric_median_np(D, weights=None, iters=200, tol=1e-7, eps=1e-8):
    D = np.asarray(D, dtype=np.float64)
    w = np.ones(len(D)) if weights is None else np.asarray(weights, dtype=np.float64)
    w = w / w.sum(); y = (w[:, None] * D).sum(axis=0)
    for _ in range(iters):
        a = w / np.maximum(np.linalg.norm(D - y, axis=1), eps)
        y_new = (a[:, None] * D).sum(axis=0) / a.sum()
        done = np.linalg.norm(y_new - y) <= tol * max(1.0, np.linalg.norm(y)); y = y_new
        if done: break
    return y

class AdaptiveServerOptimizerV2:
    STRATS = {'FedAvg', 'TrimmedMean', 'GeoMedian', 'FedAdam', 'FedYogi'}
    def __init__(self, model_shape, strategy='FedAvg', robust=None, trim_beta=0.2,
                 beta1=0.9, beta2=0.99, eta=0.1, tau=1e-3, server_lr=1.0):
        assert strategy in self.STRATS and robust in (None, 'TrimmedMean', 'GeoMedian')
        self.strategy, self.robust, self.trim_beta = strategy, robust, trim_beta
        self.beta1, self.beta2, self.eta, self.tau, self.server_lr = beta1, beta2, eta, tau, server_lr
        self.m_t = torch.zeros(model_shape); self.v_t = torch.ones(model_shape) * (tau ** 2)

    def _combine(self, D, sizes, rule):
        if rule == 'TrimmedMean': return trimmed_mean_np(D, self.trim_beta)
        if rule == 'GeoMedian':   return geometric_median_np(D, sizes)
        w = np.asarray(sizes, dtype=np.float64); w = w / w.sum()
        return (w[:, None] * D).sum(axis=0)                                           

    def aggregate(self, global_model, client_deltas, client_sizes):
        flat_global = flat_of(global_model)
        D = np.stack([d.detach().cpu().numpy() for d in client_deltas]).astype(np.float64)
        rule = self.strategy if self.strategy in ('TrimmedMean', 'GeoMedian') else self.robust
        delta = torch.tensor(self._combine(D, client_sizes, rule), dtype=torch.float32)
        if self.strategy in ('FedAvg', 'TrimmedMean', 'GeoMedian'):
            return flat_global + self.server_lr * delta
        self.m_t = self.beta1 * self.m_t + (1.0 - self.beta1) * delta
        sq = delta ** 2
        if self.strategy == 'FedYogi': self.v_t = self.v_t - (1.0 - self.beta2) * sq * torch.sign(self.v_t - sq)
        else:                          self.v_t = self.beta2 * self.v_t + (1.0 - self.beta2) * sq
        return flat_global + self.eta * (self.m_t / (torch.sqrt(self.v_t) + self.tau))

class AdvFcNN(nn.Module):
    """Replaces the 10-layer funnel: no 2-unit ReLU bottleneck (dead-ReLU risk), LayerNorm (safe under FL), logits out."""
    def __init__(self, input_dim=14, hidden=(128, 64, 32)):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden: layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.GELU()]; prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def make_xy(df, domain_flag=True):
    X = df[FEATURES].values.astype(np.float32)
    if domain_flag: X = np.concatenate([X, (df['source_domain'].values == 'UNSW').astype(np.float32)[:, None]], axis=1)
    return torch.tensor(X), torch.tensor(df['label'].values.astype(np.float32)).unsqueeze(1)

def client_update_v2(model, global_flat, df, local_epochs=2, lr=2e-3, batch_size=256, mu=0.0, domain_flag=True):
    load_flat(model, global_flat); model.train()
    anchors = [p.detach().clone() for p in model.parameters()]
    X, y = make_xy(df, domain_flag)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    opt, crit = optim.Adam(model.parameters(), lr=lr), nn.BCEWithLogitsLoss()
    for _ in range(local_epochs):
        for xb, yb in loader:
            opt.zero_grad(); loss = crit(model(xb), yb)
            if mu > 0: loss = loss + 0.5 * mu * sum(((p - a) ** 2).sum() for p, a in zip(model.parameters(), anchors))   # FedProx
            loss.backward(); opt.step()
    return flat_of(model) - global_flat

@torch.no_grad()
def predict_proba_adv(model, df, domain_flag=True, batch=16384):
    model.eval(); X, _ = make_xy(df, domain_flag)
    return np.concatenate([torch.sigmoid(model(X[i:i + batch])).numpy().ravel() for i in range(0, len(X), batch)])

## Advanced federated loop (FedProx + robust aggregation + DP + INT8)

In [20]:
def run_advanced_fl(parts, val_df, eval_df, strategy='FedAvg', robust=None, rounds=20, local_epochs=2, lr=2e-3, batch_size=256,
                    mu=0.01, clip_norm=None, dp_sigma=0.01, quantize=True, byzantine=(), byz_scale=5.0, max_rows=20000,
                    seed=SEED, domain_flag=True, verbose=True, tag=''):
    set_seed(seed)
    gm = AdvFcNN(input_dim=len(FEATURES) + (1 if domain_flag else 0))
    server = AdaptiveServerOptimizerV2(sum(p.numel() for p in gm.parameters()), strategy=strategy, robust=robust)
    dp = LocalDPEngine(clip_norm, dp_sigma, seed=seed) if clip_norm else None
    quant = Int8StochasticQuantizer(seg_sizes(gm), seed=seed) if quantize else None
    up_bytes = fp32_equiv = 0; history = []
    for r in range(1, rounds + 1):
        g_flat = flat_of(gm); deltas, sizes = [], []
        for cid in CLIENT_IDS:
            df = parts[cid].sample(n=min(max_rows, len(parts[cid])), replace=False, random_state=seed + r)
            d = client_update_v2(copy.deepcopy(gm), g_flat, df, local_epochs, lr, batch_size, mu, domain_flag).numpy()
            fp32_equiv += 4 * d.size
            if cid in byzantine:
                sent = -byz_scale * d; up_bytes += 4 * d.size
            else:
                if dp is not None: d = dp.privatize(d)
                if quant is not None:
                    payload = quant.quantize(d); up_bytes += quant.nbytes(payload); d = quant.dequantize(payload)
                else: up_bytes += 4 * d.size
                sent = d
            deltas.append(torch.tensor(sent, dtype=torch.float32)); sizes.append(len(df))
        load_flat(gm, server.aggregate(gm, deltas, sizes))
        p_val = predict_proba_adv(gm, val_df, domain_flag)
        history.append(dict(round=r, **{k: round(v, 4) for k, v in binary_metrics(val_df['label'].values, p_val >= 0.5).items()}))
        if verbose and (r % 4 == 0 or r == rounds): print(f'  [{tag}] round {r:>2} | val acc {history[-1]["acc"]:.4f} | val F1 {history[-1]["f1"]:.4f}')
    p_ev = predict_proba_adv(gm, eval_df, domain_flag)
    res = dict(model=gm, history=history, uplink_bytes=up_bytes, uplink_fp32_equiv=fp32_equiv,
               eps=(dp.epsilon(1e-5, rounds) if dp else None), domain_flag=domain_flag)
    res['eval'] = binary_metrics(eval_df['label'].values, p_ev >= 0.5, p_ev)
    return res

t0 = time.time()
adv_main = run_advanced_fl(parts_adv, val_fuse, df_eval_clean, strategy='FedAvg', mu=0.01, tag='adv FedAvg+FedProx+INT8')
print('ADVANCED classifier (FedAvg+FedProx+INT8) eval:', {k: round(v, 4) for k, v in adv_main['eval'].items()}, f'({time.time()-t0:.0f}s)')
print('uplink: %.1f KB sent vs %.1f KB if FP32  ->  %.1f%% saving' % (adv_main['uplink_bytes'] / 1e3, adv_main['uplink_fp32_equiv'] / 1e3,
      100 * (1 - adv_main['uplink_bytes'] / adv_main['uplink_fp32_equiv'])))

  [adv FedAvg+FedProx+INT8] round  4 | val acc 0.9084 | val F1 0.9215
  [adv FedAvg+FedProx+INT8] round  8 | val acc 0.9044 | val F1 0.9171
  [adv FedAvg+FedProx+INT8] round 12 | val acc 0.9133 | val F1 0.9252
  [adv FedAvg+FedProx+INT8] round 16 | val acc 0.9162 | val F1 0.9276
  [adv FedAvg+FedProx+INT8] round 20 | val acc 0.9192 | val F1 0.9304
ADVANCED classifier (FedAvg+FedProx+INT8) eval: {'acc': 0.9191, 'f1': 0.9303, 'prec': 0.9919, 'rec': 0.8759, 'auc': np.float64(0.9931)} (116s)
uplink: 1535.2 KB sent vs 6113.8 KB if FP32  ->  74.9% saving
